# Berkeley CS182 · Fall 2026 · Homework 02
## Accelerating Gradient Descent with Momentum: eigenvalue visualization

This companion notebook visualizes the two eigenvalues of the recurrence matrix
in the momentum problem. It uses the same numerical values as the assignment:
`beta = 3/4`, singular values `2` and `1`, and default learning rate `eta = 1/3`.

Run all cells in order in Colab or Jupyter. The code is provided in full;
no code completion or notebook submission is required. The animation includes
playback controls and a frame slider, and runs without installing a video encoder.

Fall 2026 contributors: Matteo Guarrera, Mert Cemri, Sizhe Chen.

Original problem and visualization credits: Suhong Moon, Gabriel Goh, Anant Sahai,
Peter Wang, Yuxi Liu.


### What is being plotted?

For each singular value $\sigma_i$, the worksheet recurrence is

$$
\begin{bmatrix}a_{t+1}[i]\\x_{t+1}[i]\end{bmatrix}
=
\begin{bmatrix}
1-\beta & 2\beta\sigma_i^2\\
-\eta(1-\beta) & 1-2\eta\beta\sigma_i^2
\end{bmatrix}
\begin{bmatrix}a_t[i]\\x_t[i]\end{bmatrix}.
$$

The dots show this matrix's eigenvalues in the complex plane. The dashed circle
has radius $1$, and the green circle has radius $\sqrt{1-\beta}=1/2$.
The left and right panels use the same learning rate and correspond to
$\sigma=2$ and $\sigma=1$, respectively. A repeated pair is shown at one location
and labeled **double root**.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML, display

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 13,
    "axes.titlesize": 16,
    "axes.labelsize": 13,
    "figure.facecolor": "white",
    "savefig.facecolor": "white",
    "animation.embed_limit": 30,
})

BETA = 3 / 4
SINGULAR_VALUES = (2.0, 1.0)
ETA_DEFAULT = 1 / 3

def eigenvalues(eta, sigma, beta=BETA):
    """Eigenvalues of the worksheet's [a_t[i], x_t[i]] recurrence."""
    trace = 2 - beta - 2 * eta * beta * sigma**2
    determinant = 1 - beta
    discriminant = trace**2 - 4 * determinant
    if np.isclose(discriminant, 0.0, atol=1e-12, rtol=0.0):
        discriminant = 0.0
    root = np.lib.scimath.sqrt(discriminant)
    return np.array([(trace + root) / 2, (trace - root) / 2], dtype=complex)

def root_type(roots):
    if np.isclose(roots[0], roots[1], atol=1e-12, rtol=0.0):
        return "Repeated real roots"
    if np.allclose(roots.imag, 0.0, atol=1e-12, rtol=0.0):
        return "Distinct real roots"
    return "Complex conjugate roots"

def make_figure():
    fig, axes = plt.subplots(1, 2, figsize=(12.4, 5.5))
    fig.subplots_adjust(left=0.055, right=0.99, bottom=0.24,
                        top=0.79, wspace=0.23)
    return fig, axes

def draw_eigenvalues(fig, axes, eta, eta_label=None):
    theta = np.linspace(0, 2 * np.pi, 361)
    for ax, sigma, color in zip(axes, SINGULAR_VALUES, ("#D55E00", "#0072B2")):
        ax.clear()
        roots = eigenvalues(eta, sigma)
        ax.plot(np.cos(theta), np.sin(theta), color="#555555", linestyle="--",
                linewidth=1.6, label="Unit circle")
        radius = np.sqrt(1 - BETA)
        ax.plot(radius * np.cos(theta), radius * np.sin(theta), color="#009E73",
                linewidth=2.2, label="Radius 1/2")
        ax.axhline(0, color="#B8BDC4", linewidth=0.8)
        ax.axvline(0, color="#B8BDC4", linewidth=0.8)
        ax.scatter(roots.real, roots.imag, s=120, color=color,
                   edgecolors="white", linewidths=0.8, zorder=5)
        if root_type(roots) == "Repeated real roots":
            ax.annotate("double root", (roots[0].real, 0), xytext=(0, 22),
                        textcoords="offset points", ha="center", color=color,
                        fontsize=12)
        ax.set_aspect("equal", adjustable="box")
        ax.set_xlim(-1.8, 1.25)
        ax.set_ylim(-1.25, 1.25)
        ax.set_xticks([-1.5, -1.0, -0.5, 0.0, 0.5, 1.0])
        ax.set_yticks([-1.0, -0.5, 0.0, 0.5, 1.0])
        ax.set_xlabel("Real part")
        ax.set_ylabel("Imaginary part")
        ax.grid(alpha=0.16)
        ax.set_title(rf"$\sigma={sigma:g}$" + "\n" + root_type(roots), pad=9)
        radius_max = np.max(np.abs(roots))
        if radius_max < 1 - 1e-12:
            stability = "inside the unit circle"
        elif radius_max > 1 + 1e-12:
            stability = "a root outside the unit circle"
        else:
            stability = "a root on the unit circle"
        ax.text(0.5, -0.25, rf"Largest magnitude: {radius_max:.3f}" + "\n" + stability,
                ha="center", va="top", transform=ax.transAxes, fontsize=12)
    eta_text = eta_label if eta_label is not None else f"{eta:.4f}"
    fig.suptitle(rf"Momentum eigenvalues: $\eta={eta_text}$, $\beta=3/4$",
                 fontsize=21, y=0.99)
    if fig.legends:
        fig.legends[0].remove()
    fig.legend(*axes[0].get_legend_handles_labels(), loc="upper center",
               bbox_to_anchor=(0.5, 0.91), ncol=2, frameon=False, fontsize=12)

def plot_eigenvalues(eta=ETA_DEFAULT, eta_label=None):
    fig, axes = make_figure()
    draw_eigenvalues(fig, axes, eta, eta_label)
    return fig


### Four learning rates

The following views use `eta = 1/48`, `1/6`, `1/3`, and `1/2`.
They show how real, repeated, and complex eigenvalues appear as the learning rate
changes. The highlighted `eta = 1/3` choice matches the numerical example in the
assignment.


In [ ]:
# Four examples used in the written assignment's companion figure.
EXAMPLES = [
    (1 / 48, "1/48"),
    (1 / 6, "1/6"),
    (1 / 3, "1/3"),
    (1 / 2, "1/2"),
]
for eta, label in EXAMPLES:
    fig = plot_eigenvalues(eta, label)
    plt.show()
    plt.close(fig)


### Explore a chosen learning rate

The cell below starts at the assignment's `eta = 1/3`. You may change the value
and rerun it. Other useful transition values are `1/24` and `3/8` for
the `sigma = 2` direction, and `1/6` and `3/2` for the `sigma = 1` direction.
If you use a value above `1/2`, some eigenvalues can leave the displayed window.


In [ ]:
fig = plot_eigenvalues(eta=ETA_DEFAULT, eta_label="1/3")
plt.show()
plt.close(fig)


### Watch the eigenvalues move

This animation increases `eta` from `1/48` to `1/2`, keeping `beta = 3/4` and
both singular values fixed. The axes stay fixed so the two directions remain
easy to compare. Rendering the inline animation can take a few seconds.


In [ ]:
# This animation is displayed inline. It does not write image frames or need ffmpeg.
# The replay slider lets you inspect individual learning rates.
eta_values = np.linspace(1 / 48, 1 / 2, 73)
fig, axes = make_figure()

def update(frame):
    draw_eigenvalues(fig, axes, float(eta_values[frame]))

animation = FuncAnimation(fig, update, frames=len(eta_values),
                          interval=100, repeat=True)
animation_html = animation.to_jshtml(fps=10)
plt.close(fig)
display(HTML(animation_html))


### Numerical verification

The following provided checks compare the plotted values with the eigenvalues
computed directly from the recurrence matrix.


In [ ]:
# Sanity checks: the plotted roots agree with the worksheet matrix.
for eta, _ in EXAMPLES:
    for sigma in SINGULAR_VALUES:
        R = np.array([
            [1 - BETA, 2 * BETA * sigma**2],
            [-eta * (1 - BETA), 1 - 2 * eta * BETA * sigma**2],
        ])
        roots = eigenvalues(eta, sigma)
        assert np.allclose(np.sort_complex(roots),
                           np.sort_complex(np.linalg.eigvals(R)))
        assert np.allclose(np.prod(roots), 1 - BETA)
        assert np.allclose(np.sum(roots), np.trace(R))

for sigma in SINGULAR_VALUES:
    assert np.allclose(np.abs(eigenvalues(ETA_DEFAULT, sigma)), 1 / 2)
    for eta in (1 / (6 * sigma**2), 3 / (2 * sigma**2)):
        assert root_type(eigenvalues(eta, sigma)) == "Repeated real roots"
print("Checks passed: recurrence eigenvalues, products, and repeated-root endpoints.")
